In [ ]:
import os
import sys
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# Project root = MTinv_OT (parent of this experiment folder)
_script_dir = os.getcwd()
_project_root = os.path.dirname(_script_dir)
if _project_root not in sys.path:
    sys.path.insert(0, _project_root)

import torch
import numpy as np
import matplotlib.pyplot as plt
from src.mt2d_inv import MT2DInverterWeightedCost
from src.mt2d_inv.io import ExperimentLogger
from pathlib import Path

logger = ExperimentLogger(
    model_tag="AKBST-AMT-L08",
    output_root=Path("test_results"),
)

os.environ["CUDA_VISIBLE_DEVICES"] = "1"
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")


In [ ]:
# ============================================================
# 0) 全局画图参数（统一管理，一改全改）
# ============================================================
PLOT_CONFIG = {
    "mc_cmap": "jet_r",
    "mc_clip_to_stations": True,
    "mc_ylim": [10, 0],
    "mc_profile_extend_km": 5.0,
    "mc_profile_axis_width_km": 45.0,
    "mc_vmin": 0.0,
    "mc_vmax": 4.0,
    "mc_axes_aspect": 1.07,
    "im_cmap": "jet_r",
    "im_clip_to_stations": True,
    "im_ylim": [10, 0],
    "im_profile_extend_km": 5.0,
    "im_profile_axis_width_km": 50.0,
    "df_station_indices": "all",
    "df_plot_noise_cap": None,
    "pf_depth_limit_km": 10,
}

save_plot_kwargs = PLOT_CONFIG


In [ ]:
from __future__ import annotations

from src.mt2d_inv.data_prep import PrepareData

# ================= USER PARAMETERS =================
EDI_DIR = "AKBST-AMT-L08"
FREQ_MIN_HZ = 1
FREQ_MAX_HZ = 1000
MAG_DECLINATION_DEG = 4
# ===================================================

prep = PrepareData(
    edi_dir=EDI_DIR,
    n_freq_target=20,
    mag_declination_deg=MAG_DECLINATION_DEG,
    edi_impedance_unit="mv/km/nt",
    freq_min_hz=FREQ_MIN_HZ,
    freq_max_hz=FREQ_MAX_HZ,
)

prep.run_all_simple()


In [ ]:
# 1. 垂向网格
nza = 10
z_air = -np.logspace(np.log10(10), np.log10(50000), nza)
z_air = np.flip(z_air)
z_air = np.append(z_air, 0)

nz = 70
z_sub = np.logspace(np.log10(5), np.log10(25000), nz)
zn = np.concatenate([z_air[:-1], np.array([0]), z_sub])

# 2. 横向网格
y_center = np.linspace(-25000, 25000, 231)
y_left = -np.logspace(np.log10(25500), np.log10(70000), 20)
y_right = np.logspace(np.log10(25500), np.log10(70000), 20)
y_left = np.flip(y_left)
yn = np.concatenate([y_left, y_center, y_right])

print("新的网格统计:")
print(f" - 垂向第一层厚度: {zn[nza+1] - zn[nza]:.1f} m")
print(f" - 最大深度: {zn[-1]/1000:.1f} km")
print(f" - 中心区横向分辨率: {y_center[1] - y_center[0]:.1f} m")
print(f" - 网格总数: {len(zn)-1} x {len(yn)-1}")


In [ ]:
freqs_t, stations_t, data_dict = prep.export_data_dict_for_2d_inversion(device=device)

inv = MT2DInverterWeightedCost(
    yn=torch.as_tensor(yn, dtype=torch.float64, device=device),
    zn=torch.as_tensor(zn, dtype=torch.float64, device=device),
    nza=nza,
    freqs=freqs_t,
    stations=stations_t,
    device=device,
    data_loss_scale=200,
)
inv.load_obs_data(data_dict, noise_floor=0.05)

w_d_point = inv.update_ot_w_d_per_point_from_noise(
    w_d_scale=[5, 1, 1, 1],
    normalize="mean",
)
print("per-point w_d:", tuple(w_d_point.shape), "mean per dim:", w_d_point.mean(dim=0).detach().cpu().numpy())


In [ ]:
inv.set_forward_operator()
inv.initialize_model(initial_sigma=0.002)
inv.print_ot_dimension_contributions()

inv.plot_initial_model(
    clip_to_stations=PLOT_CONFIG["im_clip_to_stations"],
    ylim=PLOT_CONFIG["im_ylim"],
    profile_extend_km=PLOT_CONFIG["im_profile_extend_km"],
    profile_axis_width_km=PLOT_CONFIG["im_profile_axis_width_km"],
)

final_sigma = inv.run_inversion(
    n_epochs=300,
    mode="6dot",
    progress_interval=10,
    current_lambda=1,
    use_adaptive_lambda=True,
    lr=0.05,
    update_interval=20,
    norm_type="L2",
    alpha=0.8,
    use_depth_weights=True,
    rms_chi2_stop=1.05,
    enable_blur_anneal=True,
)

print("反演完成，最终模型范围:")
print(final_sigma.min().item(), final_sigma.max().item())


In [ ]:
inv.plot_loss_history()

inv.plot_model_comparison(
    cmap=PLOT_CONFIG["mc_cmap"],
    clip_to_stations=PLOT_CONFIG["mc_clip_to_stations"],
    ylim=PLOT_CONFIG["mc_ylim"],
    profile_extend_km=PLOT_CONFIG["mc_profile_extend_km"],
    profile_axis_width_km=PLOT_CONFIG["mc_profile_axis_width_km"],
    vmin=PLOT_CONFIG["mc_vmin"],
    vmax=PLOT_CONFIG["mc_vmax"],
    axes_aspect=PLOT_CONFIG["mc_axes_aspect"],
)

print(freqs_t.min().item(), freqs_t.max().item())
batch_size = 3
n_stations = len(inv.stations)
for start in range(0, n_stations, batch_size):
    end = min(start + batch_size, n_stations)
    station_indices = list(range(start, end))
    inv.plot_data_fitting(station_indices=station_indices)
inv.plot_1d_profiles(depth_limit_km=PLOT_CONFIG["pf_depth_limit_km"])
inv.plot_gradient_history()


In [ ]:
logger.save_from_inverter(
    inv,
    run_name="ot-TE5",
    plot_kwargs=save_plot_kwargs,
)
